In [1]:
# Install dependencies first
!pip install ipywidgets
!pip install mne
!pip install numpy matplotlib scipy
!pip install python-picard


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 48.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for python-picard: filename=python_picard-0.8-py3-none-any.whl size=16394 sha256=6e397952f059a8e8b6514ad6b2721c9344848227485a9aa46ac39e981e583d14
  Stored in directory: /root/.cache/pip/wheels/86/7e/fc/6183c6bfdc791d9e7b5a1df84ec4f30055f76bc8414ab2300d
Successfully built python-picard


In [2]:
import pandas as pd
import numpy as np
import mne
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, Text, VBox, HBox, Checkbox, IntSlider
from scipy import signal
from scipy.signal import decimate

In [3]:
fileUpload = Text(value='data.bdf', description='File Path:')
downsampleOnOff = Checkbox(value=True, description='Downsample')
# downsampleDropdown = Dropdown(options=['128', '256', '512'], value='128', description='Downsample Rate:')
downsamplebar = IntSlider(value=2, min=1, max=10, step=1, description='Factor Samplerate:', orientation='horizontal')
bandpassOnOff = Checkbox(value=True, description='Bandpass Filter')
icaMethod = Dropdown(options=['fastica', 'infomax', 'picard'], value='picard', description='ICA Method:')
icaOnOff = Checkbox(value=True, description='ICA')
ui = VBox([
    fileUpload,
    HBox([downsampleOnOff, downsamplebar]),
    bandpassOnOff,
    HBox([icaOnOff, icaMethod])
])
display(ui)

# Layered Architecture



### Dataloader Class

In [4]:
class Dataloader:
  def convertBdfToCvs(self, dataToCopy):
    bdfToCsvRaw = dataToCopy.copy()
    channels = bdfToCsvRaw.ch_names[:32]  #THIS IS A BRUTE FORCE FOR DEAP
    bdfToCsvRaw.pick(channels)
    bdfToCsvData, bdfToCsvTimes = bdfToCsvRaw.get_data(return_times=True)

    channelsSanitized = ["RAW_" + ch for ch in channels]   # Sanitize channel names
    df = pd.DataFrame(bdfToCsvData.T, columns=channelsSanitized) # Transpose so each row is a timestamp, each column is a channel
    df.insert(0, 'Time', bdfToCsvTimes)  # Add a Time column (seconds)
    newCsvPath = f'/content/convertedFile.csv' # Save to CSV

    print("Creating File")
    df.to_csv(newCsvPath, index=False) #ADD timeit bar
    print(f"Saved CSV to: {newCsvPath}")
    return newCsvPath

  def getSampleRate(self,times ):
    diffs = np.diff(times)
    dt = np.median(diffs)
    sfreq = round(1 / dt)
    return sfreq

  def timeStampToSeconds(self, df, sampleRate):
      # overwrite the Time/Timestamp column using the sample rate
      df['Time'] = np.arange(len(df)) * (1 / sampleRate)


  def getData(self, fileUpload, sampleRate):
      if fileUpload.endswith('.bdf'):
          raw = mne.io.read_raw_bdf(fileUpload, preload=True).pick('eeg')
          fileUpload = self.convertBdfToCvs(raw)

      df = pd.read_csv(fileUpload, on_bad_lines='error')

      columns_to_keep = [df.columns[0]] + [col for col in df.columns if "RAW_" in col] #only keeo columns of RAW eeg data
      df = df[columns_to_keep]
      df = df.dropna()

      # Force time to be regenerated using provided sample rate
      self.timeStampToSeconds(df, sampleRate)

      eegData = df.iloc[:, 1:].values.T
      channels = df.columns[1:].tolist()

      info = mne.create_info(ch_names=channels, sfreq=sampleRate, ch_types='eeg')
      raw = mne.io.RawArray(eegData, info)
      return raw




### PreProcessing Class



In [5]:
class Preprocessing:
  def __init__(self, icaMethod, randSeed, nComponents = 32):
    self.ica = mne.preprocessing.ICA(nComponents, random_state=randSeed, max_iter='auto', method = icaMethod.value)

  def preprocess(self, rawEegData, downsampleTrue, downsamplerate, bandpassTrue, icaTrue, icaMethod):

    #BandPass Filter
    if(bandpassTrue == True):
      rawEegBandpass = rawEegData.copy().filter(l_freq=4.0, h_freq=45.0) #can add in extra feature for bandpass range | range 4 - 45 comes from the bandpass filter that deap use
      if(icaOnOff.value == True): #ICA with bandpass
        self.ica.fit(rawEegBandpass)
      eegData, times = rawEegBandpass.get_data(return_times=True)
    else:
      if(icaOnOff.value == True): #ICA without bandpass
        self.ica.fit(rawEegData) # Pass the raw object instead of raw.get_data
      eegData, times = rawEegData.get_data(return_times=True) # Get data from rawEegData

    #Downsampling
    dataArray = eegData
    if(downsampleTrue == True):
      factor = downsamplebar.value                                                 #This needs automisation, can get data sample rate with raw.info['sfreq']
      downsampled_data = np.zeros((len(dataArray), len(dataArray[0]) // factor))
      for i in range(len(dataArray)):
        downsampled_data[i] = decimate(dataArray[i], factor, ftype='fir')
      dataArray = downsampled_data

    return dataArray

### Feature Extraction Class

In [6]:
from scipy.fft import fft, fftfreq, rfftfreq, rfft

class featureExtraction:

  bands = {   #from national library of medicine: https://www.ncbi.nlm.nih.gov/books/NBK539805/   SETTING POTENTIAL
    'delta': (0.5, 4),
    'theta': (4, 7),
    'alpha': (7, 12),
    'beta':  (12, 30),
    'gamma': (30, 80)
  }
  def __init__(self, sampleRate):
    self.sR = sampleRate

  def runFFT(self, data): #runs the FFT on a 1D array that is 'data'
    yf = np.abs(rfft(data))**2  # gets power spectrum
    xf = rfftfreq (len(data), 1 / self.sR)
    return xf, yf

  def fftFreqBands(self, xfreq, power, band): #seperates the fft data into respective frequency bands for increased model performance
    lowRange, highRange = self.bands[band]
    idx = np.where((xfreq >= lowRange) & (xfreq <= highRange))
    return np.sum(power[idx]) #can change this to np.mean      SETTING POTENTIAL?

  def getAllBands(self, data):
    xf, power = self.runFFT(data)
    return {band: self.fftFreqBands(xfreq=xf, power=power, band=band) for band in self.bands}

  def plotAllBands(self, data):
    xf, power = self.runFFT(data)

    plt.figure(figsize=(12, 6))
    for i, (band, (low, high)) in enumerate(self.bands.items()):
        idx = np.where((xf >= low) & (xf <= high))
        band_freq = xf[idx]
        band_power = power[idx]

        # Normalize band power to [0, 1]
        if np.max(band_power) != np.min(band_power):
            norm_power = (band_power - np.min(band_power)) / (np.max(band_power) - np.min(band_power))
        else:
            norm_power = band_power  # Flat line, don't normalize

        plt.plot(band_freq, norm_power, label=f"{band} ({low}-{high} Hz)", linewidth=2)

    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Normalized Power')
    plt.title('Normalized FFT Power Spectrum by Band')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

  def plotBandsOverTime(self, data, window_size_sec=1.0, overlap=0.5): #Generated by AI to assist in visualising the data is correctly transforming
    step_size = int(window_size_sec * self.sR * (1 - overlap))
    window_len = int(window_size_sec * self.sR)
    num_steps = (len(data) - window_len) // step_size + 1

    times = []
    band_powers = {band: [] for band in self.bands}

    for i in range(num_steps):
        start = i * step_size
        end = start + window_len
        segment = data[start:end]

        if len(segment) < window_len:
            continue

        xf, power = self.runFFT(segment)
        times.append(start / self.sR)

        for band in self.bands:
            band_power = self.fftFreqBands(xf, power, band)
            band_powers[band].append(band_power)

    for band in band_powers:
        arr = np.array(band_powers[band])
        if arr.max() != arr.min():
            band_powers[band] = (arr - arr.min()) / (arr.max() - arr.min())
        else:
            band_powers[band] = arr  # Flat line

    # Plotting
    plt.figure(figsize=(12, 6))
    for band in self.bands:
        plt.plot(times, band_powers[band], label=band, linewidth=2)

    plt.xlabel('Time (s)')
    plt.ylabel('Normalized Band Power')
    plt.title('EEG Band Power Over Time')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

  def getBandsOverTime(self, data, window_size_sec=1.0, overlap=0.5):
    step_size = int(window_size_sec * self.sR * (1 - overlap))
    window_len = int(window_size_sec * self.sR)
    num_steps = (len(data) - window_len) // step_size + 1

    times = []
    band_powers = {band: [] for band in self.bands}

    for i in range(num_steps):
        start = i * step_size
        end = start + window_len
        segment = data[start:end]

        if len(segment) < window_len:
            continue

        xf, power = self.runFFT(segment)
        times.append(start / self.sR)

        for band in self.bands:
            band_power = self.fftFreqBands(xf, power, band)
            band_powers[band].append(band_power)

    for band in band_powers:
        arr = np.array(band_powers[band])
        if arr.max() != arr.min():
            band_powers[band] = (arr - arr.min()) / (arr.max() - arr.min())
        else:
            band_powers[band] = arr
    return times, band_powers

  def extractFeatures(self, preprocessedData):
    # all_windows = []
    # bands = list(self.bands.keys())  # ['delta', 'theta', 'alpha', 'beta', 'gamma']
    # n_channels = preprocessedData.shape[0]

    # for ch in range(n_channels):
    #     channel_data = preprocessedData[ch]
    #     channel_features = self.getBandsOverTime(channel_data)  # e.g. list of lists of floats

    #     for i, window_vals in enumerate(channel_features):
    #         # Convert list of floats to dict with band names
    #         window_dict = dict(zip(bands, window_vals))

    #         # Prefix keys with channel number to avoid key collisions
    #         prefixed = {f'ch{ch}_{k}': v for k, v in window_dict.items()}

    #         if len(all_windows) <= i:
    #             all_windows.append(prefixed)
    #         else:
    #             all_windows[i].update(prefixed)

    # df = pd.DataFrame(all_windows)
    data = []
    for ch in range(preprocessedData.shape[0]):
        channel_windows = self.getBandsOverTime(preProcessedData[ch])
        # Optionally, add channel info to each window dict

    return channel_windows

### Classification Class


In [7]:
features

NameError: name 'features' is not defined

### Waterfall Code


In [ ]:
RANDSEED = 52
SAMPLERATE = 256 #256 for mindmonitor, 512 for DEAP
dataloader = Dataloader()
preProcessor = Preprocessing(randSeed = RANDSEED, nComponents = 0.999999, icaMethod = icaMethod) #the 0.9999 provides most consistent clear ICA results across different data, thus why used
featureExtactor = featureExtraction(SAMPLERATE)

raw1 = dataloader.getData(fileUpload.value, sampleRate = SAMPLERATE) # [channels, time samples]
preProcessedData = preProcessor.preprocess(rawEegData = raw1, downsampleTrue = downsampleOnOff.value,downsamplerate = downsamplebar, bandpassTrue = bandpassOnOff.value, icaTrue = icaOnOff.value, icaMethod = icaMethod) #(channels, samples)
features = featureExtactor.extractFeatures(preProcessedData)




# Testing and Running


In [ ]:
features

In [ ]:
plt.plot(features)

In [ ]:
preProcessedData[5]

# Feature Extraction


In [ ]:
from scipy.fft import fft, fftfreq
testFE = preProcessedData[1]
# Number of samples in normalized_tone
N = int(SAMPLERATE * (len(testFE) / SAMPLERATE))

yf = fft(testFE)
xf = fftfreq(N, 1 / SAMPLERATE)

plt.plot(xf, np.abs(yf))
plt.show()

In [ ]:
plt.plot(xf[:N//2], abs(yf[:N//2]))
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.title("FFT of Downsampled Signal")
plt.grid()
plt.show()

In [ ]:
fftData = fE.getAllBands(testFE)
fE.plotBandsOverTime(testFE, 1, 0.5)
